In [ ]:
# ============== ШАБЛОН НА ВСЕ СЛУЧАИ ЖИЗНИ ==============

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import joblib

# 1. Загрузить данные
df = pd.read_csv('ваш_файл.csv', encoding='utf-8', sep=',')

# 2. Быстрая очистка
# Удаляем столбцы с >50% пропусков
df = df.loc[:, df.isna().mean() < 0.5]
# Заполняем оставшиеся пропуски медианой
df = df.fillna(df.median(numeric_only=True))

# 3. Разделить данные
X = df.drop(columns=['target_column'])  # заменить на ваш target
y = df['target_column']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Обучить модель (самую простую и надежную)
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 5. Оценить
y_pred = model.predict(X_test)
print(f"R²: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

# 6. Сохранить модель
joblib.dump(model, 'model.joblib')
print("Модель сохранена как 'model.joblib'")

In [ ]:
test

In [ ]:
# Загрузка данных
def load_and_prepare_data(file_path, encoding='utf-8', sep=','):
    """
    Шаблон загрузки данных с первичной обработкой
    """
    # 1. Чтение
    df = pd.read_csv(file_path, encoding=encoding, sep=sep)
    
    # 2. Первичный анализ
    print("Размер данных:", df.shape)
    print("\nТипы данных:")
    print(df.dtypes)
    
    # 3. Обработка пропусков (строковые значения)
    # Если есть "-", "не определено" и т.п.
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col].replace(['-', 'не определено', 'NaN', ''], np.nan), errors='coerce')
    
    # 4. Удаление неинформативных признаков
    cols_to_drop = []
    
    # А) Уникальные идентификаторы
    for col in df.columns:
        if df[col].nunique() == len(df):
            cols_to_drop.append(col)
            print(f"Удален уникальный идентификатор: {col}")
    
    # Б) Признаки с одним значением
    for col in df.columns:
        if df[col].nunique() == 1:
            cols_to_drop.append(col)
            print(f"Удален признак с одним значением: {col}")
    
    # В) Признаки с >80% пропусков
    for col in df.columns:
        missing_percent = df[col].isna().sum() / len(df) * 100
        if missing_percent > 80:
            cols_to_drop.append(col)
            print(f"Удален признак с {missing_percent:.1f}% пропусков: {col}")
    
    # Удаляем
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
    
    # 5. Разделение на train/test
    from sklearn.model_selection import train_test_split
    
    # Целевая переменная (последний столбец или указать явно)
    target_col = df.columns[-1]  # или указать вручную: 'target'
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
    
    return X_train, X_test, y_train, y_test

# Использование:
# X_train, X_test, y_train, y_test = load_and_prepare_data('data.csv', encoding='windows-1251', sep='|')

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# Шаблон функции для Optuna
def optimize_model(model_class, X_train, y_train, n_trials=20, cv_folds=3):
    """
    Универсальная функция для оптимизации гиперпараметров
    """
    def objective(trial):
        params = {}
        
        # Линейные модели
        if model_class.__name__ == 'LinearRegression':
            params = {}
        
        elif model_class.__name__ == 'Ridge':
            params['alpha'] = trial.suggest_float('alpha', 0.001, 100.0, log=True)
            params['random_state'] = 42
            
        elif model_class.__name__ == 'Lasso':
            params['alpha'] = trial.suggest_float('alpha', 0.001, 10.0, log=True)
            params['max_iter'] = trial.suggest_categorical('max_iter', [1000, 5000, 10000])
            params['random_state'] = 42
            
        # Случайный лес
        elif model_class.__name__ == 'RandomForestRegressor':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 30),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                'random_state': 42,
                'n_jobs': -1
            }
        
        # Градиентный бустинг
        elif model_class.__name__ == 'GradientBoostingRegressor':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 200),
                'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.5, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 7),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'random_state': 42
            }
        
        # SVR
        elif model_class.__name__ == 'SVR':
            params = {
                'C': trial.suggest_float('C', 0.1, 100.0, log=True),
                'epsilon': trial.suggest_float('epsilon', 0.01, 0.5),
                'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly']),
                'max_iter': 10000
            }
        
        # Создаем модель
        model = model_class(**params)
        
        # Кросс-валидация
        cv = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
        scores = cross_val_score(model, X_train, y_train, cv=cv, 
                                 scoring='neg_mean_squared_error', n_jobs=-1)
        
        return scores.mean()
    
    # Оптимизация
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    # Лучшая модель
    best_params = study.best_params
    
    if model_class.__name__ not in ['LinearRegression']:
        if 'random_state' not in best_params:
            best_params['random_state'] = 42
    
    best_model = model_class(**best_params)
    best_model.fit(X_train, y_train)
    
    print(f"Лучшие параметры: {best_params}")
    print(f"Лучший MSE (CV): {-study.best_value:.6f}")
    
    return best_model, best_params

# Шаблон оценки моделей
def evaluate_model(model, X_train, X_test, y_train, y_test):
    """
    Оценка модели на train и test
    """
    # Предсказания
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Метрики
    metrics = {}
    
    for name, y_true, y_pred in [('Train', y_train, y_train_pred), 
                                 ('Test', y_test, y_test_pred)]:
        metrics[f'{name}_MSE'] = mean_squared_error(y_true, y_pred)
        metrics[f'{name}_MAE'] = mean_absolute_error(y_true, y_pred)
        metrics[f'{name}_R2'] = r2_score(y_true, y_pred)
    
    # Вывод
    print(f"{'Метрика':<15} {'Train':<10} {'Test':<10}")
    print("-" * 35)
    for metric in ['MSE', 'MAE', 'R2']:
        print(f"{metric:<15} {metrics[f'Train_{metric}']:.6f}  {metrics[f'Test_{metric}']:.6f}")
    
    return metrics

# Пример использования:
# models = [LinearRegression, Ridge, RandomForestRegressor]
# for model_class in models:
#     print(f"\n{'='*50}")
#     print(f"Оптимизация {model_class.__name__}")
#     best_model, params = optimize_model(model_class, X_train, y_train, n_trials=10)
#     evaluate_model(best_model, X_train, X_test, y_train, y_test)

In [ ]:
# Классификация
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Регрессия
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# Кластеризация
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture

In [ ]:
# dashboard.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score

# Настройка страницы
st.set_page_config(
    page_title="ML Dashboard",
    page_icon="🤖",
    layout="wide"
)

# Заголовок
st.title("🤖 ML Dashboard для прогнозирования")
st.markdown("---")

# Сайдбар
st.sidebar.header("Настройки")
uploaded_file = st.sidebar.file_uploader("Загрузите CSV файл", type="csv")

# Разделы
tab1, tab2, tab3, tab4 = st.tabs(["📁 Данные", "📊 Анализ", "🤖 Модель", "📈 Прогноз"])

with tab1:
    st.header("Загрузка и просмотр данных")
    
    if uploaded_file is not None:
        # Чтение данных
        df = pd.read_csv(uploaded_file)
        st.success(f"Файл загружен! Размер: {df.shape[0]} строк, {df.shape[1]} столбцов")
        
        # Показать данные
        st.subheader("Первые 10 строк")
        st.dataframe(df.head(10))
        
        # Сохранить в сессии
        st.session_state['df'] = df
        
        # Основные статистики
        st.subheader("Основные статистики")
        st.dataframe(df.describe())
        
    else:
        st.info("Загрузите CSV файл через сайдбар")

with tab2:
    st.header("Визуальный анализ данных")
    
    if 'df' in st.session_state:
        df = st.session_state['df']
        
        col1, col2 = st.columns(2)
        
        with col1:
            # Гистограмма выбранного признака
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            selected_col = st.selectbox("Выберите признак для гистограммы", numeric_cols)
            
            fig, ax = plt.subplots()
            ax.hist(df[selected_col].dropna(), bins=30, edgecolor='black')
            ax.set_title(f"Распределение {selected_col}")
            st.pyplot(fig)
        
        with col2:
            # Корреляционная матрица
            st.subheader("Корреляционная матрица")
            if len(numeric_cols) > 1:
                corr_matrix = df[numeric_cols].corr()
                fig, ax = plt.subplots(figsize=(10, 8))
                sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
                st.pyplot(fig)
            else:
                st.warning("Недостаточно числовых признаков для матрицы корреляций")

with tab3:
    st.header("Работа с ML моделью")
    
    # Загрузка модели
    model_loaded = st.checkbox("Загрузить предобученную модель")
    
    if model_loaded:
        model_file = st.file_uploader("Загрузите модель (.joblib или .pkl)", type=['joblib', 'pkl'])
        
        if model_file:
            model = joblib.load(model_file)
            st.session_state['model'] = model
            st.success("Модель загружена!")
            
            # Информация о модели
            st.subheader("Информация о модели")
            st.code(f"Тип модели: {type(model).__name__}")
            
            if hasattr(model, 'feature_importances_'):
                st.write("Важность признаков:")
                importance_df = pd.DataFrame({
                    'Признак': X_train.columns if 'X_train' in locals() else [f'Feature_{i}' for i in range(len(model.feature_importances_))],
                    'Важность': model.feature_importances_
                }).sort_values('Важность', ascending=False)
                st.dataframe(importance_df)

with tab4:
    st.header("Прогнозирование")
    
    if 'model' in st.session_state and 'df' in st.session_state:
        model = st.session_state['model']
        df = st.session_state['df']
        
        # Выбор признаков
        st.subheader("Выбор признаков для прогноза")
        
        # Автоматически выбираем числовые признаки
        features = df.select_dtypes(include=[np.number]).columns.tolist()
        
        if 'target' in features:
            features.remove('target')
        
        # Создаем DataFrame для предсказания
        X_pred = df[features].copy()
        
        # Кнопка предсказания
        if st.button("Запустить прогноз"):
            with st.spinner("Выполняется прогноз..."):
                predictions = model.predict(X_pred)
                
                # Результаты
                st.success("Прогноз выполнен!")
                
                # Таблица с результатами
                results_df = pd.DataFrame({
                    'ID': range(len(predictions)),
                    'Предсказание': predictions
                })
                
                st.dataframe(results_df.head(20))
                
                # Визуализация
                fig, ax = plt.subplots()
                ax.hist(predictions, bins=30, edgecolor='black', alpha=0.7)
                ax.set_xlabel("Значение прогноза")
                ax.set_ylabel("Количество")
                ax.set_title("Распределение прогнозов")
                st.pyplot(fig)
                
                # Скачать результаты
                csv = results_df.to_csv(index=False)
                st.download_button(
                    label="Скачать прогнозы (CSV)",
                    data=csv,
                    file_name="predictions.csv",
                    mime="text/csv"
                )
    else:
        st.warning("Сначала загрузите данные и модель")

# Запуск в Colab с ngrok:
"""
1. Установить:
!pip install streamlit pyngrok

2. Запустить:
from pyngrok import ngrok
!streamlit run dashboard.py &>/dev/null &
public_url = ngrok.connect(8501)
print(f"Дашборд доступен по ссылке: {public_url}")
"""

In [ ]:
# =================== ЭКЗАМЕН: STREAMLIT ДАШБОРД ===================
# Сохранить как: exam_dashboard.py
# Запустить в терминале: streamlit run exam_dashboard.py

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# ========== НАСТРОЙКА ==========
st.set_page_config(page_title="Экзамен DS", layout="wide")
st.title("📊 Экзаменационный дашборд")
st.markdown("---")

# ========== 1. ЗАГРУЗКА ДАННЫХ ==========
st.header("📁 1. Загрузка данных")
uploaded_file = st.file_uploader("Загрузите CSV файл", type=["csv"])

if uploaded_file is not None:
    # Чтение файла
    try:
        df = pd.read_csv(uploaded_file)
    except:
        df = pd.read_csv(uploaded_file, encoding='windows-1251', sep='|')
    
    st.success(f"✅ Загружено: {df.shape[0]} строк, {df.shape[1]} столбцов")
    
    # ========== 2. ПРОСМОТР ДАННЫХ ==========
    tab1, tab2, tab3 = st.tabs(["Данные", "Анализ", "Модель"])
    
    with tab1:
        st.subheader("Просмотр данных")
        st.dataframe(df.head(10))
        
        col1, col2 = st.columns(2)
        with col1:
            st.metric("Строк", df.shape[0])
        with col2:
            st.metric("Столбцов", df.shape[1])
    
    with tab2:
        st.subheader("Анализ данных")
        
        # Информация о типах
        st.write("**Типы данных:**")
        dtype_info = pd.DataFrame({
            'Столбец': df.columns,
            'Тип': df.dtypes.values,
            'Пропуски': df.isna().sum().values
        })
        st.dataframe(dtype_info)
        
        # Статистики
        if st.checkbox("Показать статистики"):
            st.dataframe(df.describe())
        
        # Визуализация
        st.write("**Визуализация:**")
        num_cols = df.select_dtypes(include=np.number).columns
        if len(num_cols) > 0:
            selected_col = st.selectbox("Выберите столбец", num_cols)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.hist(df[selected_col].dropna(), bins=20, edgecolor='black')
            ax.set_title(f"Распределение: {selected_col}")
            st.pyplot(fig)
    
    with tab3:
        st.subheader("Обучение модели")
        
        # Выбор целевой переменной
        target_options = list(df.columns)
        target_col = st.selectbox("Выберите целевую переменную", target_options)
        
        if target_col:
            # Подготовка данных
            X = df.drop(columns=[target_col])
            y = df[target_col]
            
            # Только числовые признаки
            X = X.select_dtypes(include=np.number)
            
            if X.shape[1] > 0:
                # Параметры
                test_size = st.slider("Размер тестовой выборки (%)", 10, 40, 20)
                n_trees = st.slider("Количество деревьев", 10, 200, 100)
                
                if st.button("🎯 Обучить модель", type="primary"):
                    with st.spinner("Обучение..."):
                        # Разделение
                        X_train, X_test, y_train, y_test = train_test_split(
                            X, y, test_size=test_size/100, random_state=42
                        )
                        
                        # Обучение
                        model = RandomForestRegressor(
                            n_estimators=n_trees,
                            random_state=42,
                            n_jobs=-1
                        )
                        model.fit(X_train, y_train)
                        
                        # Прогноз
                        y_pred = model.predict(X_test)
                        
                        # Метрики
                        r2 = r2_score(y_test, y_pred)
                        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                        
                        # Вывод
                        st.success("✅ Модель обучена!")
                        
                        col1, col2, col3 = st.columns(3)
                        with col1:
                            st.metric("R²", f"{r2:.4f}")
                        with col2:
                            st.metric("RMSE", f"{rmse:.4f}")
                        with col3:
                            st.metric("Объектов", X.shape[0])
                        
                        # График
                        fig, ax = plt.subplots(figsize=(8, 6))
                        ax.scatter(y_test, y_pred, alpha=0.5)
                        ax.plot([y_test.min(), y_test.max()], 
                               [y_test.min(), y_test.max()], 'r--')
                        ax.set_xlabel("Факт")
                        ax.set_ylabel("Прогноз")
                        st.pyplot(fig)
                        
                        # Сохраняем в сессии
                        st.session_state['model'] = model
                        st.session_state['features'] = X.columns.tolist()
            else:
                st.warning("❌ Нет числовых признаков для обучения")
    
    # ========== 3. ПРОГНОЗ НОВЫХ ДАННЫХ ==========
    st.header("🔮 3. Прогнозирование")
    
    if 'model' in st.session_state:
        model = st.session_state['model']
        
        # Загрузка новых данных
        new_file = st.file_uploader("Загрузите новые данные для прогноза", type=["csv"])
        
        if new_file:
            try:
                new_df = pd.read_csv(new_file)
            except:
                new_df = pd.read_csv(new_file, encoding='windows-1251', sep='|')
            
            # Проверка признаков
            expected = st.session_state['features']
            available = [col for col in expected if col in new_df.columns]
            
            if len(available) == len(expected):
                if st.button("📊 Выполнить прогноз"):
                    predictions = model.predict(new_df[expected])
                    
                    # Результаты
                    results = pd.DataFrame({
                        'ID': range(len(predictions)),
                        'Прогноз': predictions
                    })
                    
                    st.dataframe(results)
                    
                    # Скачать
                    csv = results.to_csv(index=False)
                    st.download_button(
                        "📥 Скачать прогнозы",
                        csv,
                        "predictions.csv",
                        "text/csv"
                    )
            else:
                st.warning(f"❌ Нужны признаки: {expected[:5]}...")
    else:
        st.info("👈 Сначала обучите модель")
    
else:
    st.info("👈 Загрузите CSV файл для начала работы")

# ========== ФУТЕР ==========
st.markdown("---")
st.caption("📌 Для экзамена по Data Science | Запуск: streamlit run exam_dashboard.py")